# PD logistic regression and held-out validation

This notebook models the probability that an account is good. Probability of default (PD) is calculated explicitly as `1 - P(good)`. It consumes the engineered train/test matrices from notebooks 00–03.

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from scipy import stats

processed = Path('../data/processed')

## Load the model matrices and targets

The files below are produced by the preceding notebooks. They are intentionally loaded rather than rebuilt here.

In [ ]:
with open(processed / 'model_inputs_train.pkl', 'rb') as f:
    inputs_train_all = pickle.load(f)
with open(processed / 'model_inputs_test.pkl', 'rb') as f:
    inputs_test_all = pickle.load(f)
with open(processed / 'targets_train.pkl', 'rb') as f:
    targets_train = pickle.load(f)
with open(processed / 'targets_test.pkl', 'rb') as f:
    targets_test = pickle.load(f)

targets_train = pd.Series(targets_train).astype(int)
targets_test = pd.Series(targets_test).astype(int)

## Original candidate inventory and reference levels

This is the full candidate inventory selected in the original model notebook. One level per dummy family is omitted as its reference so coefficients remain identifiable.

In [ ]:
candidate_features = '''grade:A grade:B grade:C grade:D grade:E grade:F grade:G home_ownership:RENT_OTHER_NONE_ANY home_ownership:OWN home_ownership:MORTGAGE addr_state:ND_NE_IA_NV_FL_HI_AL addr_state:NM_VA addr_state:NY addr_state:OK_TN_MO_LA_MD_NC addr_state:CA addr_state:UT_KY_AZ_NJ addr_state:AR_MI_PA_OH_MN addr_state:RI_MA_DE_SD_IN addr_state:GA_WA_OR addr_state:WI_MT addr_state:TX addr_state:IL_CT addr_state:KS_SC_CO_VT_AK_MS addr_state:WV_NH_WY_DC_ME_ID verification_status:Not_Verified verification_status:Source_Verified verification_status:Verified purpose:educ__sm_b__wedd__ren_en__mov__house purpose:credit_card purpose:debt_consolidation purpose:oth__med__vacation purpose:major_purch__car__home_impr initial_list_status:f initial_list_status:w term:36 term:60 emp_length:0 emp_length:1 emp_length:2-4 emp_length:5-6 emp_length:7-9 emp_length:10 mths_since_issue_d:<38 mths_since_issue_d:38-39 mths_since_issue_d:40-41 mths_since_issue_d:42-48 mths_since_issue_d:49-52 mths_since_issue_d:53-64 mths_since_issue_d:65-84 mths_since_issue_d:>84 int_rate:<9.548 int_rate:9.548-12.025 int_rate:12.025-15.74 int_rate:15.74-20.281 int_rate:>20.281 mths_since_earliest_cr_line:<140 mths_since_earliest_cr_line:141-164 mths_since_earliest_cr_line:165-247 mths_since_earliest_cr_line:248-270 mths_since_earliest_cr_line:271-352 mths_since_earliest_cr_line:>352 delinq_2yrs:0 delinq_2yrs:1-3 delinq_2yrs:>=4 inq_last_6mths:0 inq_last_6mths:1-2 inq_last_6mths:3-6 inq_last_6mths:>6 open_acc:0 open_acc:1-3 open_acc:4-12 open_acc:13-17 open_acc:18-22 open_acc:23-25 open_acc:26-30 open_acc:>=31 pub_rec:0-2 pub_rec:3-4 pub_rec:>=5 total_acc:<=27 total_acc:28-51 total_acc:>=52 acc_now_delinq:0 acc_now_delinq:>=1 total_rev_hi_lim:<=5K total_rev_hi_lim:5K-10K total_rev_hi_lim:10K-20K total_rev_hi_lim:20K-30K total_rev_hi_lim:30K-40K total_rev_hi_lim:40K-55K total_rev_hi_lim:55K-95K total_rev_hi_lim:>95K annual_inc:<20K annual_inc:20K-30K annual_inc:30K-40K annual_inc:40K-50K annual_inc:50K-60K annual_inc:60K-70K annual_inc:70K-80K annual_inc:80K-90K annual_inc:90K-100K annual_inc:100K-120K annual_inc:120K-140K annual_inc:>140K dti:<=1.4 dti:1.4-3.5 dti:3.5-7.7 dti:7.7-10.5 dti:10.5-16.1 dti:16.1-20.3 dti:20.3-21.7 dti:21.7-22.4 dti:22.4-35 dti:>35 mths_since_last_delinq:Missing mths_since_last_delinq:0-3 mths_since_last_delinq:4-30 mths_since_last_delinq:31-56 mths_since_last_delinq:>=57 mths_since_last_record:Missing mths_since_last_record:0-2 mths_since_last_record:3-20 mths_since_last_record:21-31 mths_since_last_record:32-80 mths_since_last_record:81-86 mths_since_last_record:>86'''.split()
# Correction: source columns use spaces in two verification labels; resolve them from the checkpoint.
candidate_features = [name.replace('_', ' ') if name.startswith('verification_status:') else name for name in candidate_features]
reference_categories_initial = ['grade:G', 'home_ownership:RENT_OTHER_NONE_ANY', 'addr_state:ND_NE_IA_NV_FL_HI_AL', 'verification_status:Verified', 'purpose:educ__sm_b__wedd__ren_en__mov__house', 'initial_list_status:f', 'term:60', 'emp_length:0', 'mths_since_issue_d:>84', 'int_rate:>20.281', 'mths_since_earliest_cr_line:<140', 'delinq_2yrs:>=4', 'inq_last_6mths:>6', 'open_acc:0', 'pub_rec:0-2', 'total_acc:<=27', 'acc_now_delinq:0', 'total_rev_hi_lim:<=5K', 'annual_inc:<20K', 'dti:>35', 'mths_since_last_delinq:0-3', 'mths_since_last_record:0-2']
missing = sorted(set(candidate_features) - set(inputs_train_all.columns))
if missing:
    raise KeyError(f'Candidate columns missing from model checkpoint: {missing}')
X_initial = inputs_train_all[candidate_features].drop(columns=reference_categories_initial)
X_initial.shape

### Candidate-family inventory

The inventory spans grade, home ownership, address state, verification, purpose, listing status, term, employment, issue timing, interest rate, credit history, delinquency, inquiry, account-count, revolving-limit, income, debt-to-income, and missingness-aware delinquency/record bands.

In [ ]:
pd.Series(candidate_features).str.split(':').str[0].value_counts().sort_index().rename('category_count')

## Initial model and p-values

P-values use the observed Fisher information of the fitted unpenalized coefficient estimates. They are descriptive diagnostics; the regularized classifier remains the prediction model.

In [ ]:
def coefficient_summary(model, X):
    probabilities = model.predict_proba(X)[:, 1]
    weights = probabilities * (1 - probabilities)
    information = X.to_numpy(dtype=float).T @ (weights[:, None] * X.to_numpy(dtype=float))
    standard_errors = np.sqrt(np.diag(np.linalg.pinv(information)))
    z_scores = model.coef_[0] / standard_errors
    return pd.DataFrame({'feature': X.columns, 'coefficient': model.coef_[0], 'p_value': 2 * stats.norm.sf(np.abs(z_scores))})

initial_model = LogisticRegression(solver='liblinear', max_iter=1000)
initial_model.fit(X_initial, targets_train)
initial_summary = coefficient_summary(initial_model, X_initial)
pd.concat([pd.DataFrame({'feature': ['Intercept'], 'coefficient': initial_model.intercept_, 'p_value': [np.nan]}), initial_summary], ignore_index=True)

## Final specification

The original review removed families whose dummy levels were all or almost all non-significant: delinquencies, open accounts, public records, total accounts, and revolving-limit bands. This retains the broader original final specification rather than the later five-feature simplification.

In [ ]:
removed_families = ('delinq_2yrs:', 'open_acc:', 'pub_rec:', 'total_acc:', 'total_rev_hi_lim:')
final_candidate_features = [f for f in candidate_features if not f.startswith(removed_families)]
reference_categories = [f for f in reference_categories_initial if f in final_candidate_features]
X_train = inputs_train_all[final_candidate_features].drop(columns=reference_categories)
X_test = inputs_test_all[final_candidate_features].drop(columns=reference_categories)
final_model = LogisticRegression(solver='liblinear', max_iter=1000)
final_model.fit(X_train, targets_train)
final_summary = coefficient_summary(final_model, X_train)
model_summary = pd.concat([pd.DataFrame({'feature': ['Intercept'], 'coefficient': final_model.intercept_, 'p_value': [np.nan]}), final_summary], ignore_index=True)
model_summary

### Final coefficient review

This table provides the final intercept, category coefficients, and diagnostic p-values used to document the retained specification.

In [ ]:
model_summary.sort_values(['feature']).reset_index(drop=True)

## Held-out predictions

`P(good)` is the probability of target class 1. The PD column is its complement; no probability is relabelled implicitly.

In [ ]:
p_good = final_model.predict_proba(X_test)[:, 1]
validation_predictions = pd.DataFrame({'actual_good': targets_test.to_numpy(), 'p_good': p_good})
validation_predictions['pd'] = 1 - validation_predictions['p_good']
validation_predictions.head()

## Threshold analysis

Accuracy can look favorable in this class-imbalanced target, so it is reported alongside the count and proportion confusion matrices rather than as a stand-alone assessment.

In [ ]:
thresholds_to_review = [0.5, 0.7, 0.9]
threshold_results = []
for threshold in thresholds_to_review:
    predicted_good = (validation_predictions['p_good'] >= threshold).astype(int)
    matrix = confusion_matrix(validation_predictions['actual_good'], predicted_good, labels=[0, 1])
    threshold_results.append({'threshold': threshold, 'accuracy': (predicted_good == validation_predictions['actual_good']).mean(), 'confusion_matrix': matrix})
threshold_results

In [ ]:
threshold = 0.5
validation_predictions['predicted_good'] = (validation_predictions['p_good'] >= threshold).astype(int)
confusion_counts = pd.crosstab(validation_predictions['actual_good'], validation_predictions['predicted_good'], rownames=['actual'], colnames=['predicted'])
confusion_proportions = confusion_counts / len(validation_predictions)
display(confusion_counts)
display(confusion_proportions)
print(f'Accuracy at P(good) >= {threshold:.1f}: {(validation_predictions.actual_good == validation_predictions.predicted_good).mean():.3f}')

## ROC, AUC, Gini, and KS

These rank-based measures describe discrimination across thresholds. They complement, rather than replace, the threshold results above.

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(validation_predictions['actual_good'], validation_predictions['p_good'])
auc = roc_auc_score(validation_predictions['actual_good'], validation_predictions['p_good'])
gini = 2 * auc - 1
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], '--', color='black')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve for P(good)')
plt.legend()
plt.show()
print(f'Gini coefficient: {gini:.3f}')

In [ ]:
cumulative = validation_predictions.sort_values('p_good').reset_index(drop=True).copy()
cumulative['cumulative_population'] = (cumulative.index + 1) / len(cumulative)
cumulative['cumulative_good'] = cumulative['actual_good'].cumsum() / cumulative['actual_good'].sum()
cumulative['cumulative_bad'] = (1 - cumulative['actual_good']).cumsum() / (1 - cumulative['actual_good']).sum()
ks = (cumulative['cumulative_bad'] - cumulative['cumulative_good']).max()
plt.plot(cumulative['cumulative_population'], cumulative['cumulative_bad'], label='bad')
plt.plot(cumulative['cumulative_population'], cumulative['cumulative_good'], label='good')
plt.xlabel('Cumulative population')
plt.ylabel('Cumulative share')
plt.title(f'KS curve (KS = {ks:.3f})')
plt.legend()
plt.show()
ks

## Save the modeling handoff

The saved model predicts `P(good)`; downstream scorecard work derives PD as `1 - P(good)`.

In [ ]:
processed.mkdir(parents=True, exist_ok=True)
with open(processed / 'pd_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)
with open(processed / 'model_feature_names.pkl', 'wb') as f:
    pickle.dump(list(X_train.columns), f)
with open(processed / 'reference_categories.pkl', 'wb') as f:
    pickle.dump(reference_categories, f)
with open(processed / 'validation_predictions.pkl', 'wb') as f:
    pickle.dump(validation_predictions, f)

## Conclusions

The held-out results should be interpreted as a combination of ranking evidence (AUC, Gini, KS) and operational threshold behavior. A high accuracy at a single threshold may mostly reflect the prevalence of good accounts; the ROC and cumulative curves expose the model's discrimination across thresholds.